# Entregas Finais — Projeto Aplicado MCD

**Instituto SENAI de Tecnologia Ambiental — UniSENAI**

Este notebook demonstra cada entrega solicitada pelo projeto, com o código correspondente do `pipeline.py` e os resultados obtidos.

In [ ]:
import pandas as pd
import numpy as np
import requests
import time
import re
import json
from urllib.parse import quote
import warnings
warnings.filterwarnings('ignore')

session = requests.Session()
TIMEOUT = 10
DELAY = 0.05

---
## 1. Juntar Planilha de Identificação e Abundância

Merge das planilhas `IDENTIFICACAO.xlsx` e `ABUND.xlsx` pela coluna `Compound` (inner join), com limpeza de duplicatas e registros sem identificação.

In [ ]:
df_id = pd.read_excel('dados_brutos/IDENTIFICACAO.xlsx')
df_ab = pd.read_excel('dados_brutos/ABUND.xlsx')
df_id.columns = df_id.columns.str.strip()
df_ab.columns = df_ab.columns.str.strip()

print(f'IDENTIFICACAO: {df_id.shape[0]} linhas × {df_id.shape[1]} colunas')
print(f'ABUND: {df_ab.shape[0]} linhas × {df_ab.shape[1]} colunas')

# Limpeza
if 'Compound ID' not in df_id.columns:
    df_id['Compound ID'] = df_id['Compound']
df_id = df_id.drop_duplicates(subset=['Compound']).dropna(subset=['Description'])
df_ab = df_ab.drop_duplicates(subset=['Compound'])

# Merge
df_merged = pd.merge(df_id, df_ab, on='Compound', how='inner')
print(f'\nApós merge: {df_merged.shape[0]} linhas × {df_merged.shape[1]} colunas')
print(f'Compostos únicos: {df_merged["Description"].nunique()}')
df_merged[['Compound', 'Description', 'Score', 'Formula']].head(5)

---
## 2. Verificação: Produto Natural/Metabólito ou Composto Sintético

Classificação baseada na ontologia ChEBI e heurísticas por nome. Regras em ordem de prioridade:
1. Peptídeos detectados por abreviações de aminoácidos no nome
2. Termos da ontologia ChEBI (fatty acid → Natural, drug → Sintético, etc.)
3. Fallback por nome (contém "acid") ou primeiro parent da ontologia

In [ ]:
AMINOACIDOS = ['ala','arg','asn','asp','cys','gln','glu','gly','his','ile',
               'leu','lys','met','phe','pro','ser','thr','trp','tyr','val']

REGRAS_ONTOLOGIA = [
    (['fatty acid','lipid','acyl'], 'Ácido graxo / Lipídio', 'Lipídios', 'Natural'),
    (['amino acid'], 'Aminoácido', 'Aminoácidos', 'Natural'),
    (['carbohydrate','sugar','glycoside'], 'Carboidrato / Glicosídeo', 'Carboidratos', 'Natural'),
    (['terpene','terpenoid','isoprenoid'], 'Terpenoide', 'Terpenos', 'Natural'),
    (['alkaloid'], 'Alcaloide', 'Alcaloides', 'Natural'),
    (['flavonoid','phenol','polyphenol'], 'Flavonoide / Polifenol', 'Fenólicos', 'Natural'),
    (['steroid','sterol'], 'Esteroide', 'Lipídios', 'Natural'),
    (['nucleotide','nucleoside','purine','pyrimidine'], 'Nucleotídeo', 'Nucleotídeos', 'Natural'),
    (['vitamin'], 'Vitamina', 'Cofatores', 'Natural'),
    (['organic acid','carboxylic acid'], 'Ácido orgânico', 'Metabolismo primário', 'Natural'),
    (['drug','pharmaceutical','xenobiotic'], 'Fármaco / Xenobiótico', 'Xenobiótico', 'Sintético'),
]

def classificar_composto(nome, ontologia=''):
    nome_lower = (nome or '').lower()
    ontologia = ontologia.lower()
    # Peptídeos
    partes = nome_lower.replace('-',' ').split()
    n_amino = sum(1 for p in partes if p in AMINOACIDOS)
    if n_amino >= 2:
        nomes_pep = {2:'dipeptídeo',3:'tripeptídeo',4:'tetrapeptídeo',5:'pentapeptídeo'}
        return 'Natural', f'Peptídeo ({nomes_pep.get(n_amino, str(n_amino)+" aa")})'
    # Ontologia
    for palavras, cat, met, tipo in REGRAS_ONTOLOGIA:
        if any(w in ontologia for w in palavras):
            return tipo, cat
    # Fallback
    if 'acid' in nome_lower:
        return 'Natural', 'Ácido orgânico'
    if ontologia:
        return 'Natural', ontologia.split(' | ')[0]
    return 'Indeterminado', 'Não classificado'

# Demonstração
testes = [
    ('gln-pro-trp-tyr', ''),
    ('Sertraline', 'secondary amino compound | organic amino compound'),
    ('Citric acid', ''),
    ('Aspirin', 'drug | pharmaceutical'),
]
for nome, ont in testes:
    tipo, cat = classificar_composto(nome, ont)
    print(f'  {nome:25s} → {tipo:15s} | {cat}')

---
## 3. Consulta a Bases de Dados Públicas (PubChem e ChEBI)

Consultamos duas bases:
- **PubChem PUG REST** — propriedades moleculares (CID, fórmula, massa, IUPAC)
- **ChEBI** (via PubChem Synonyms + Classification) — ID ChEBI e hierarquia taxonômica

In [ ]:
def normalizar_nome(nome):
    if pd.isna(nome): return None
    nome = str(nome).split('[')[0].strip()
    nome = re.sub(r'[^a-zA-Z0-9\s\-\(\),]', '', nome)
    nome = re.sub(r'\s+', ' ', nome)
    if len(nome) > 80: return None
    if nome.lower() in ['unknown','unidentified','na','null']: return None
    return nome

def buscar_pubchem(nome):
    if not nome: return {}
    for tentativa in [nome, nome.split(',')[0], ' '.join(nome.split()[:4])]:
        url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{tentativa}/property/MolecularFormula,MolecularWeight,IUPACName/JSON'
        try:
            r = session.get(url, timeout=TIMEOUT)
            if r.status_code == 200:
                p = r.json()['PropertyTable']['Properties'][0]
                return {'pubchem_cid': p.get('CID'), 'formula': p.get('MolecularFormula'),
                        'massa_api': p.get('MolecularWeight'), 'iupac': p.get('IUPACName')}
        except: pass
    return {}

def buscar_chebi_via_pubchem(cid):
    if not cid: return {}
    result = {}
    try:
        # ChEBI ID via sinônimos
        r = session.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{int(cid)}/synonyms/JSON', timeout=TIMEOUT)
        if r.status_code == 200:
            for s in r.json().get('InformationList',{}).get('Information',[{}])[0].get('Synonym',[]):
                if 'CHEBI:' in s.upper():
                    result['chebi_id'] = s; break
        time.sleep(DELAY)
        # Ontologia via classificação
        r2 = session.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{int(cid)}/classification/JSON', timeout=15)
        if r2.status_code == 200:
            for h in r2.json().get('Hierarchies',{}).get('Hierarchy',[]):
                if h.get('SourceName') != 'ChEBI': continue
                nodes = h.get('Node',[])
                def gn(n):
                    o=n.get('Information',{}).get('Name',{})
                    return o.get('StringWithMarkup',{}).get('String','') if isinstance(o,dict) else str(o)
                for n in nodes:
                    if n.get('Information',{}).get('Match'):
                        result['chebi_nome']=gn(n); mid=n.get('NodeID'); break
                else: break
                nm={n.get('NodeID'):n for n in nodes}; ps=[]; c=nm.get(mid)
                for _ in range(5):
                    if not c: break
                    pids=c.get('ParentID',[])
                    if not pids: break
                    pid=pids[0] if isinstance(pids,list) else pids
                    pn=nm.get(pid)
                    if not pn: break
                    name=gn(pn)
                    if name and name not in ['chemical entity','molecular entity']: ps.append(name)
                    c=pn
                if ps: result['ontologia']=' | '.join(ps)
                break
    except: pass
    return result

# Demonstração
for nome in ['Sertraline', 'caffeine']:
    pub = buscar_pubchem(nome)
    chebi = buscar_chebi_via_pubchem(pub.get('pubchem_cid'))
    print(f'{nome}:')
    print(f'  PubChem CID: {pub.get("pubchem_cid")} | Fórmula: {pub.get("formula")} | Massa: {pub.get("massa_api")}')
    print(f'  ChEBI ID: {chebi.get("chebi_id")} | Ontologia: {chebi.get("ontologia","—")[:80]}')
    print()

---
## 4. Obtenção da Ontologia ou Classificação Química (ChEBI)

A ontologia é obtida pela hierarquia ChEBI disponível no PubChem Classification. Subimos a árvore a partir do nó do composto (Match) até 5 níveis, excluindo nós genéricos como "chemical entity" e "molecular entity".

O resultado é a cadeia de classificação do composto: ex. `secondary amino compound | organic amino compound | organic molecular entity`

In [ ]:
# Resultado do pipeline — compostos com ontologia preenchida
df = pd.read_excel('compostos_final_resultado.xlsx')

com_ontologia = df[df['Ontologia (ChEBI)'].notna()]
print(f'Compostos com ontologia ChEBI: {len(com_ontologia)} de {len(df)}')
print()

com_ontologia[['Metabólito/Composto', 'Categoria química', 'Ontologia (ChEBI)']].head(10)

---
## 5. Levantamento de Usos e Aplicações Conhecidas (PubChem)

Utilizamos o endpoint de descrição do PubChem (`/compound/cid/{CID}/description/JSON`) para obter textos descritivos que incluem usos, aplicações farmacológicas e propriedades biológicas do composto.

In [ ]:
def buscar_pubchem_descricao(cid):
    """Busca descrição/usos do composto no PubChem."""
    if not cid: return {}
    try:
        r = session.get(f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{int(cid)}/description/JSON', timeout=TIMEOUT)
        if r.status_code != 200: return {}
        for info in r.json().get('InformationList',{}).get('Information',[]):
            desc = info.get('Description','')
            if desc and len(desc) > 20:
                return {'uso_descricao': desc[:500]}
    except: pass
    return {}

# Demonstração
for cid, nome in [(68617, 'Sertraline'), (2519, 'Caffeine')]:
    r = buscar_pubchem_descricao(cid)
    print(f'{nome} (CID {cid}):')
    print(f'  {r.get("uso_descricao", "Sem descrição")[:150]}...')
    print()

In [ ]:
# Resultado do pipeline — compostos com usos preenchidos
com_usos = df[df['Usos/Aplicações'].notna()]
print(f'Compostos com usos/aplicações: {len(com_usos)} de {len(df)}')
print()

for _, row in com_usos.head(5).iterrows():
    print(f'  {row["Metabólito/Composto"][:50]}')
    print(f'  → {str(row["Usos/Aplicações"])[:120]}...')
    print()

---
## Resumo Geral

Visão consolidada de todas as entregas.

In [ ]:
df = pd.read_excel('compostos_final_resultado.xlsx')

total = len(df)
print(f'Total de compostos únicos: {total}')
print(f'Com PubChem CID:           {df["PubChem CID"].notna().sum()}')
print(f'Com ChEBI ID:              {df["ChEBI ID"].notna().sum()}')
print(f'Com Ontologia:             {df["Ontologia (ChEBI)"].notna().sum()}')
print(f'Com Usos/Aplicações:       {df["Usos/Aplicações"].notna().sum()}')
print(f'Classificados:             {(df["Categoria química"] != "Não classificado").sum()}')
print(f'  Naturais:                {(df["Tipo (Natural/Sintético)"] == "Natural").sum()}')
print(f'  Sintéticos:              {(df["Tipo (Natural/Sintético)"] == "Sintético").sum()}')
print()
print('Colunas da planilha final:')
print(list(df.columns))

In [ ]:
df.head(10)